# 02 - Audit and clean the data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/demand-forecasting-capstone/blob/main/notebooks/02_data_audit_and_cleaning.ipynb)

**Goal:** measure the quality of the raw data, apply written cleaning rules, and build the modelling table.

**Why first?** Models are only as good as the data. This notebook finds missing dates, duplicates, zero-heavy items, store-closure days and weather gaps *before* any modelling.

How to run: `Runtime -> Run all` (Colab) or run the cells from top to bottom. The default dataset is `synthetic`, which needs no downloads.

In [ ]:
# ---- Setup: works in Google Colab and on your own computer. Nothing to upload. ----
REPO_URL = "https://github.com/YOUR-USERNAME/demand-forecasting-capstone.git"   # <- set once, the same in every notebook
BRANCH = "main"
USE_DRIVE = False   # True = use a copy of the project folder in your Google Drive instead of GitHub

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if not os.path.exists("src/demand_agents"):
    if IN_COLAB and USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.chdir("/content/drive/MyDrive/demand-forecasting-capstone")
    elif IN_COLAB:
        if not os.path.exists("/content/project"):
            subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/project"], check=True)
        os.chdir("/content/project")
    elif os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                                   # running locally from the notebooks folder
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"], check=True)
sys.path.insert(0, os.path.abspath("src"))
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)
from IPython.display import display, Image, Markdown
print("Project folder:", os.getcwd())

## 1. Pick the dataset

In [ ]:
DATASET = "synthetic"    # "synthetic" | "synthetic_calm" | "m5" | "m5_christmas" | "walmart" | "walmart_synthetic"
from demand_agents.config import load_named
cfg = load_named(DATASET)
print("dataset:", DATASET, "| source:", cfg["data"]["source"], "| frequency:", cfg["data"]["frequency"])

## 2. Raw audit

We load the data in its raw form (one row per series and period) and measure it.

In [ ]:
from demand_agents.data.loaders import load_raw_dataset
from demand_agents.data.audit import audit_panel
raw = load_raw_dataset(cfg)
a = audit_panel(raw, cfg["data"]["frequency"], DATASET, cfg["data"]["season_period"])
display(pd.Series(a["overview"], name="overview").to_frame())
display(pd.Series(a["quality"], name="quality counts").to_frame())

## 3. Issues found, and what we do about each

`error` = must be fixed, `warn` = fix or be careful, `info` = good to know.

In [ ]:
display(pd.DataFrame(a["issues"]) if a["issues"] else "no issues found")

## 4. Cleaning rules

Every rule is counted. **We deliberately keep demand spikes and zero-sales days**: they are the shocks this project studies.

In [ ]:
from demand_agents.data.clean import clean_panel
clean, log = clean_panel(raw, cfg["data"]["target_type"])
display(pd.DataFrame(log))

## 5. Weather (only if a weather source is configured)

In [ ]:
from demand_agents.data.integrate import load_weather
from demand_agents.data.clean import clean_weather
from demand_agents.data.audit import audit_weather
w = load_weather(cfg)
if w is None:
    print("No weather source for this dataset (set up NOAA in notebook 01 for M5).")
else:
    wc, wlog = clean_weather(w)
    display(pd.DataFrame(wlog))
    states = sorted(clean["store_id"].str.split("_").str[0].unique()) if DATASET.startswith("m5") else None
    wa = audit_weather(wc, clean["date"].unique(), states)
    display(pd.Series({k: v for k, v in wa.items() if k != "issues"}).to_frame("weather audit"))
    display(pd.DataFrame(wa["issues"]) if wa["issues"] else "weather looks fine")

## 6. Build the modelling table

This runs clean -> audit -> merge weather and **writes** `data/interim/` (cleaned) and `data/processed/` (the table the models read), plus the audit under `reports/tables/`.

In [ ]:
from demand_agents.data.integrate import build_dataset
summary = build_dataset(cfg)
print("processed table:", summary["processed_path"], "| rows:", f"{summary['rows_final']:,}")
display(pd.DataFrame(summary["audit_final"]["issues"]) if summary["audit_final"]["issues"] else "no remaining issues")

## 7. Two things worth knowing about demand

In [ ]:
from demand_agents.data.audit import demand_classes, mass_zero_days
cl = demand_classes(summary["table"])
display(cl["class"].value_counts().to_frame("series"))
print("Days when almost every series sold nothing (closures):", mass_zero_days(summary["table"]) or "none")

Demand types: *smooth* (steady), *erratic* (steady but variable size), *intermittent* (many zero days), *lumpy* (zeros and variable size).

**Next:** `03_eda.ipynb`.